# Part 1: Practise

## Task 1 : The mean and the median

In [1]:
import pandas as pd

date_columns = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']
listings = pd.read_csv("data/listings_clean.csv", parse_dates=date_columns)
print(listings.shape)

(4574, 73)


### Five small numbers of reviews

In [3]:
# pd.Series([...]) makes one small column from your own numbers.
small = pd.Series([3, 5, 8, 10, 12])
print("mean:", small.mean())
print("median:", small.median())

mean: 7.6
median: 8.0


### Same numbers plus 1293 (the most popular listing)

In [4]:
with_big = pd.Series([3, 5, 8, 10, 12, 1293])
print("mean:", round(with_big.mean(), 1))
print("median:", with_big.median())

mean: 221.8
median: 9.0


### Mean and median for the real column

In [5]:
print("mean:", round(listings["number_of_reviews"].mean(), 1))
print("median:", listings["number_of_reviews"].median())

mean: 35.8
median: 9.0


**Answer:** For the five numbers the mean is 7.6 and the median is 8.
When I add 1293 the mean jumps to 221.8, but the median only goes to 9. One big number pulls the mean up, but the median just looks at the middle.
For the real column the mean is 35.8 and the median is 9.
For a typical listing I would use the median, 9. Half of the listings have 9 reviews or less. The mean is too high because of a few very popular listings.

## Task 2 : Percentiles and the spread

### describe() on number_of_reviews

In [6]:
print(listings["number_of_reviews"].describe().round(1))

count    4574.0
mean       35.8
std        78.4
min         0.0
25%         2.0
50%         9.0
75%        29.0
max      1293.0
Name: number_of_reviews, dtype: float64


### Percentiles

In [7]:
print(listings["number_of_reviews"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64


### q1, q3 and the IQR

In [8]:
q1 = listings["number_of_reviews"].quantile(0.25)
q3 = listings["number_of_reviews"].quantile(0.75)
iqr = q3 - q1
print("q1:", q1, " q3:", q3, " iqr:", iqr)

q1: 2.0  q3: 29.0  iqr: 27.0


**Answer:** describe() gives count 4574, mean 35.8, std 78.4, min 0, 25% 2, median 9, 75% 29 and max 1293.
The percentiles are 0, 2, 9, 29, 94 and 400.
q1 is 2, q3 is 29, so the IQR is 27. That means the middle half of the listings have between 2 and 29 reviews.
The 10th percentile is 0, so at least 10 % of the listings have no reviews at all.
From 90 % (94) to 99 % (400) it jumps a lot, so a few listings have very many reviews.

## Task 3 : The fences and the outliers

### Upper and lower fence

In [9]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print("upper:", upper, " lower:", lower)

upper: 69.5  lower: -38.5


### How many are above the upper fence

In [10]:
above = listings[listings["number_of_reviews"] > upper]
print(len(above))
print(round(len(above) / len(listings) * 100, 1), "percent")

582
12.7 percent


### Room types of the outliers

In [11]:
print(above["room_type"].value_counts())

room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64


### Median price for the outliers and for the rest

In [12]:
others = listings[listings["number_of_reviews"] <= upper]
print("outliers:", above["price"].median())
print("others:", others["price"].median())

outliers: 1459.54
others: 1867.0


**Answer:** The upper fence is 69.5 and the lower fence is -38.5.
There can't be outliers below, because you can't have less than 0 reviews.
582 listings are above the upper fence (12.7 %). Most of them are Entire home/apt (446), then Private room 121, Shared room 14 and Hotel room 1.
The outliers have median price 1459.5 and the rest 1867. So the listings with many reviews are cheaper. Maybe cheaper places get booked more, so they get more reviews.

## Task 4 : Mark them, and compare

### Mark the outliers in a new column

In [13]:
listings["reviews_outlier"] = listings["number_of_reviews"] > upper
print(listings["reviews_outlier"].sum())

582


### Compare all listings and without outliers

In [14]:
all_rev = listings["number_of_reviews"]
without = listings[~listings["reviews_outlier"]]["number_of_reviews"]

compare = pd.DataFrame({
    "all": all_rev.agg(["count", "mean", "median", "std"]),
    "without": without.agg(["count", "mean", "median", "std"])
})
compare["change %"] = (compare["without"] - compare["all"]) / compare["all"] * 100
print(compare.round(1))

           all  without  change %
count   4574.0   3992.0     -12.7
mean      35.8     12.9     -63.8
median     9.0      7.0     -22.2
std       78.4     15.5     -80.2


**Answer:** 582 listings are marked as outliers.
With all listings: count 4574, mean 35.8, median 9, std 78.4.
Without outliers: count 3992, mean 12.9, median 7, std 15.5.
The mean goes down 64 %, the median 22 % and the std 80 %.
The std changes the most, because the outliers made the spread very big. The median changes the least, because it only looks at the middle.

## Task 5 : Describe the groups

### describe() for each room type

In [15]:
print(listings.groupby("room_type")["number_of_reviews"].describe().round(0))

                  count   mean    std   min   25%   50%    75%     max
room_type                                                             
Entire home/apt  3760.0   34.0   76.0   0.0   2.0   8.0   27.0  1293.0
Hotel room          3.0   61.0   16.0  49.0  52.0  56.0   68.0    79.0
Private room      787.0   41.0   84.0   0.0   2.0  12.0   38.0   867.0
Shared room        24.0  114.0  122.0   0.0  28.0  82.0  146.0   510.0


**Answer:** Entire home/apt has 3760 listings and median 8. Private room has 787 and median 12. Shared room has 24 and median 82. Hotel room has only 3 and median 56.
Shared room has the highest median of the groups that have enough listings.
I don't want to say anything about Hotel room, because 3 listings is too few. One more hotel could change the number a lot.

## Task 6 : Correlation

### Correlation with 7 columns

In [16]:
cols = ["number_of_reviews", "price", "accommodates", "bedrooms", "availability_365",
        "review_scores_rating", "minimum_nights", "days_since_review"]
print(listings[cols].corr()["number_of_reviews"].round(2))

number_of_reviews       1.00
price                  -0.09
accommodates           -0.05
bedrooms               -0.14
availability_365        0.18
review_scores_rating   -0.04
minimum_nights         -0.07
days_since_review      -0.19
Name: number_of_reviews, dtype: float64


### availability_365 with and without outliers

In [17]:
normal = listings[~listings["reviews_outlier"]]

print("with outliers:", listings["number_of_reviews"].corr(listings["availability_365"]).round(2))
print("without outliers:", normal["number_of_reviews"].corr(normal["availability_365"]).round(2))

with outliers: 0.18
without outliers: 0.0


**Answer:** price -0.09, accommodates -0.05, bedrooms -0.14, availability_365 0.18, rating -0.04, minimum_nights -0.07 and days_since_review -0.19.
They are all weak, close to 0.
The strongest is days_since_review (-0.19). Listings with many reviews got their last review more recently.
The weakest is rating (-0.04). All of them are negative except availability_365.
availability_365 is 0.18 with the outliers but 0.00 without them. So the 0.18 only came from the outliers.